In [1]:
import sys
import os

# Adiciona a pasta raiz do projeto aos caminhos que o Python reconhece
sys.path.append(os.path.abspath('..'))

from functools import partial
import numpy as np
from src.meamt_core import build_toolbox_knapsack, gen_inicial_tables, run, evaluate_knapsack_individual, create_knapsack_instance

os.makedirs("Resultados_Mochila_6obj", exist_ok=True)

NPOP = 1600
NGEN = 600
INDSIZE = 100
NOBJ = 6
NUM_TABLES = int((1 << NOBJ)) 
MAX_TABLE_SIZE = int(NPOP / NUM_TABLES)
REF_POINT_HV = [0.0]*NOBJ
RESET = 1000
CXPB = 0.9
MUTPB = 1.0
SEED = 42 


VALORES, PESOS, CAPACIDADES, LUCRO_MAXIMO = create_knapsack_instance(INDSIZE, NOBJ, seed=SEED)

func_eval_safe = partial(evaluate_knapsack_individual, values=VALORES, weights=PESOS, capacities=CAPACIDADES, max_profits=LUCRO_MAXIMO)
toolbox = build_toolbox_knapsack(func_eval_safe, INDSIZE, NPOP, NOBJ)
if __name__ == '__main__':
    for i in range (1, 31):
        #Gera População Inicial
        pop_ini = toolbox.population()
    
        #Avalia Indivíduos no começo
        fitnesses = toolbox.map(toolbox.evaluate, pop_ini)
        for ind, fit in zip(pop_ini, fitnesses):
            ind.fitness.values = fit
        
        tabelas = gen_inicial_tables(pop_ini, NUM_TABLES, MAX_TABLE_SIZE, NOBJ)
        pareto_real = [[0]*NOBJ]*1000
    
        logbook = run(tabelas, pareto_real, NUM_TABLES, MAX_TABLE_SIZE, NGEN, toolbox, CXPB, MUTPB, REF_POINT_HV, NOBJ, RESET)   
        
        fronteira_final = [ind.fitness.values for ind in tabelas[0]]
        matriz_resultados = np.array(fronteira_final) 
        nome_arquivo = f"Resultados_Mochila_6obj/meamt_run_{i}.npy"
        np.save(nome_arquivo, matriz_resultados)  